------------------------------------------------------------
## SECTION 1: Imports and Configuration
------------------------------------------------------------

# LLM-Based Malware Behavior Analysis

## Deliverable 3: Automated Dynamic Analysis Signature Generation

This notebook implements LLM-based analysis for:
1. Converting behavioral features into natural language descriptions
2. Using LLM to analyze malware behaviors and identify patterns
3. Generating human-readable behavioral signatures
4. Creating YARA-like detection rules from LLM analysis

**Input**: Processed features from `feature_extraction.ipynb`
**Output**: LLM-generated behavioral signatures and detection rules

In [1]:
# Import required libraries
import json
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Configure paths
PROJECT_ROOT = Path('../').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
LLM_OUTPUT_DIR = DATA_DIR / 'llm_analysis'

# Create output directory
LLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"LLM output dir: {LLM_OUTPUT_DIR}")

Project root: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation
Processed data: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\processed
LLM output dir: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\llm_analysis


------------------------------------------------------------
## SECTION 2: Load Extracted Features
------------------------------------------------------------

Load the behavioral features and signatures from the feature extraction phase:

In [2]:
# Load behavior features
behavior_features_path = PROCESSED_DIR / 'behavior_features.csv'
df_features = pd.read_csv(behavior_features_path)
print(f"✓ Loaded behavior features: {len(df_features)} samples")
print(f"  Columns: {df_features.columns.tolist()}")

# Load detailed behavior data
behavior_details_path = PROCESSED_DIR / 'behavior_details.json'
with open(behavior_details_path, 'r') as f:
    behavior_details = json.load(f)
print(f"✓ Loaded behavior details: {len(behavior_details)} samples")

# Load signatures
signatures_path = PROCESSED_DIR / 'behavioral_signatures.json'
with open(signatures_path, 'r') as f:
    signatures = json.load(f)
print(f"✓ Loaded signatures: {len(signatures)} samples")

# Load signature summary
signature_summary_path = PROCESSED_DIR / 'signature_summary.csv'
df_signatures = pd.read_csv(signature_summary_path)
print(f"✓ Loaded signature summary: {len(df_signatures)} samples")

# Display sample distribution by family if available
if 'family' in df_features.columns:
    print(f"\nSamples per family:")
    print(df_features['family'].value_counts())

# Load Behavior IR (structured intermediate representation)
behavior_ir_path = PROCESSED_DIR / 'behavior_ir.json'
behavior_ir_records = []
if behavior_ir_path.exists():
    with open(behavior_ir_path, 'r') as f:
        behavior_ir_records = json.load(f)
    print(f'✓ Loaded behavior IR: {len(behavior_ir_records)} records')
    # Build a quick lookup: sha256 -> IR record
    ir_by_hash = {r['sha256']: r for r in behavior_ir_records}
else:
    print('⚠ behavior_ir.json not found — run feature_extraction.ipynb or behavior_ir.ipynb first')
    ir_by_hash = {}

✓ Loaded behavior features: 48976 samples
  Columns: ['report_hash', 'family', 'type', 'num_api_calls', 'unique_api_count', 'api_entropy', 'num_files', 'num_write_files', 'num_delete_files', 'num_registry_keys', 'num_write_registry', 'num_delete_registry', 'num_domains', 'num_ips', 'num_http_requests', 'has_network', 'num_processes', 'num_mutexes', 'num_commands', 'has_suspicious_apis', 'has_network.1', 'drops_files', 'registry_modification', 'process_injection', 'dlopen_or_hook', 'shell_execute', 'has_file_dropper', 'has_c2_communication', 'persistence_attempt', 'behavior_count', 'num_suspicious_apis']
✓ Loaded behavior details: 48976 samples
✓ Loaded signatures: 48976 samples
✓ Loaded signature summary: 48976 samples

Samples per family:
family
Emotet      14429
Swisyn      12591
Qakbot       4895
Trickbot     4202
Lokibot      4191
njRAT        3372
Zeus         2594
Ursnif       1343
Adload        704
HarHar        655
Name: count, dtype: int64


------------------------------------------------------------
## SECTION 3: Behavior-to-Text Conversion
------------------------------------------------------------

Convert structured behavioral features into natural language descriptions for LLM analysis:

In [3]:
def behavior_to_text(sample_hash: str, features_row: pd.Series, details: dict, signature: dict) -> str:
    """
    Convert behavioral features into a structured natural language description.
    
    Args:
        sample_hash: SHA256 hash of the sample
        features_row: Row from behavior_features DataFrame
        details: Detailed behavior data from behavior_details.json
        signature: Signature data from behavioral_signatures.json
    
    Returns:
        Natural language description of the sample's behavior
    """
    
    # Build the behavioral description
    lines = []
    lines.append(f"=== MALWARE BEHAVIOR ANALYSIS REPORT ===")
    lines.append(f"Sample Hash: {sample_hash[:16]}...")
    
    # Family info if available
    if 'family' in features_row and pd.notna(features_row.get('family')):
        lines.append(f"Malware Family: {features_row['family']}")
    if 'type' in features_row and pd.notna(features_row.get('type')):
        lines.append(f"Malware Type: {features_row['type']}")
    
    lines.append(f"\n--- API ACTIVITY ---")
    lines.append(f"Total API calls: {features_row['num_api_calls']}")
    lines.append(f"Unique APIs used: {features_row['unique_api_count']}")
    
    # Top APIs from details
    if details and 'top_apis' in details:
        top_apis = details['top_apis'][:10]
        if top_apis:
            lines.append(f"Most frequent APIs: {', '.join(top_apis)}")
    
    # Suspicious APIs
    if details and 'suspicious_apis' in details:
        suspicious = details['suspicious_apis']
        if suspicious:
            lines.append(f"⚠ Suspicious APIs detected: {', '.join(suspicious[:10])}")
    
    lines.append(f"\n--- FILE SYSTEM ACTIVITY ---")
    lines.append(f"Files accessed: {features_row['num_files']}")
    lines.append(f"Files written: {features_row['num_write_files']}")
    lines.append(f"Files deleted: {features_row['num_delete_files']}")
    
    lines.append(f"\n--- REGISTRY ACTIVITY ---")
    lines.append(f"Registry keys accessed: {features_row['num_registry_keys']}")
    lines.append(f"Registry keys written: {features_row['num_write_registry']}")
    lines.append(f"Registry keys deleted: {features_row['num_delete_registry']}")
    
    lines.append(f"\n--- NETWORK ACTIVITY ---")
    lines.append(f"Domains contacted: {features_row['num_domains']}")
    lines.append(f"IP addresses: {features_row['num_ips']}")
    lines.append(f"HTTP requests: {features_row['num_http_requests']}")
    
    if details:
        if details.get('domains'):
            lines.append(f"Domains: {', '.join(details['domains'][:5])}")
        if details.get('http_requests'):
            lines.append(f"HTTP endpoints: {', '.join(details['http_requests'][:3])}")
    
    lines.append(f"\n--- PROCESS ACTIVITY ---")
    lines.append(f"Processes spawned: {features_row['num_processes']}")
    lines.append(f"Commands executed: {features_row['num_commands']}")
    lines.append(f"Mutexes created: {features_row['num_mutexes']}")
    
    if details and details.get('mutexes'):
        lines.append(f"Mutex names: {', '.join(details['mutexes'][:5])}")
    
    # Behavioral indicators from signature
    if signature:
        lines.append(f"\n--- THREAT ASSESSMENT ---")
        lines.append(f"Threat Level: {signature.get('threat_level', 'unknown').upper()}")
        
        if signature.get('detection_methods'):
            lines.append(f"Detection methods: {', '.join(signature['detection_methods'])}")
        
        if signature.get('mitre_techniques'):
            lines.append(f"MITRE ATT&CK: {', '.join(signature['mitre_techniques'])}")
        
        if signature.get('rules'):
            lines.append(f"\nDetection Rules ({len(signature['rules'])}):\n")
            for rule in signature['rules']:
                lines.append(f"  • {rule['name']}: {rule['description']}")
    
    return '\n'.join(lines)


# Test on a sample
print("Testing behavior-to-text conversion...\n")

# Find a sample with some activity
sample_idx = df_features[df_features['num_api_calls'] > 0].index[0] if len(df_features[df_features['num_api_calls'] > 0]) > 0 else 0
test_hash = df_features.iloc[sample_idx]['report_hash']
test_features = df_features.iloc[sample_idx]
test_details = next((d for d in behavior_details if d['report_hash'] == test_hash), {})
test_signature = next((s for s in signatures if s['report_hash'] == test_hash), {})

sample_text = behavior_to_text(test_hash, test_features, test_details, test_signature)
print(sample_text)

Testing behavior-to-text conversion...

=== MALWARE BEHAVIOR ANALYSIS REPORT ===
Sample Hash: 00003d128a7eb859...
Malware Family: Qakbot
Malware Type: banker

--- API ACTIVITY ---
Total API calls: 477
Unique APIs used: 477
Most frequent APIs: kernel32.dll.FlsAlloc, kernel32.dll.FlsGetValue, kernel32.dll.FlsSetValue, kernel32.dll.FlsFree, kernelbase.dll.InitializeCriticalSectionAndSpinCount, kernel32.dll.VirtualAlloc, kernel32.dll.LoadLibraryA, kernel32.dll.GetProcAddress, kernel32.dll.VirtualProtect, kernel32.dll.SetUnhandledExceptionFilter

--- FILE SYSTEM ACTIVITY ---
Files accessed: 86
Files written: 9
Files deleted: 9

--- REGISTRY ACTIVITY ---
Registry keys accessed: 208
Registry keys written: 1
Registry keys deleted: 0

--- NETWORK ACTIVITY ---
Domains contacted: 0
IP addresses: 0
HTTP requests: 0

--- PROCESS ACTIVITY ---
Processes spawned: 9
Commands executed: 11
Mutexes created: 9
Mutex names: 00003D128A7EB859F65Fa, udkyu, Global\ikkzowxr, 00003D128A7EB859F65F/C, ikkzowxra

--

------------------------------------------------------------
## SECTION 4: LLM Prompt Engineering
------------------------------------------------------------

Create structured prompts for LLM analysis of malware behaviors:

------------------------------------------------------------
## SECTION 4b: IR-Based Prompt Engineering
------------------------------------------------------------

Instead of feeding raw behavior text to the LLM, we now use the compact **Behavior IR** as the prompt input.

This makes the LLM task well-defined and testable:
```
Behavior IR  →  LLM  →  Detection Rule
```

In [ ]:
# -----------------------------------------------------------
# SECTION 4b: IR-Based Prompt Helper
# -----------------------------------------------------------

import re as _re_ir
from pathlib import Path as _Path_ir

# Motif detection constants (mirrors behavior_ir.ipynb)
_IR_INJECTION_APIS = {
    "WriteProcessMemory", "VirtualAllocEx", "CreateRemoteThread",
    "NtWriteVirtualMemory", "NtCreateThreadEx", "RtlCreateUserThread",
    "SetThreadContext", "QueueUserAPC",
}
_IR_HOOK_APIS = {
    "SetWindowsHookEx", "SetWindowsHookExA", "SetWindowsHookExW",
    "GetAsyncKeyState", "GetKeyState",
}
_IR_RUN_KEY_PATTERNS = [
    _re_ir.compile(r"\\(CurrentVersion\\Run|RunOnce|RunServices)", _re_ir.IGNORECASE),
]
_IR_EXECUTABLE_EXTS = {".exe", ".dll", ".bat", ".ps1", ".vbs", ".scr", ".com", ".pif"}


def ir_to_prompt(ir: dict) -> str:
    """
    Serialise a Behavior IR record into a compact, information-dense LLM prompt.

    The LLM's sole job is:  IR  ->  behavioral detection rule.
    This is far more reliable than feeding it thousands of raw API calls.
    """
    lines = []
    lines.append("=== BEHAVIOR IR ===")
    lines.append(f"SHA256 : {ir['sha256'][:16]}...")
    if ir.get("family") and ir["family"] != "unknown":
        lines.append(f"Family : {ir['family']}")
    if ir.get("type") and ir["type"] != "unknown":
        lines.append(f"Type   : {ir['type']}")

    lines.append("\n-- Scored Behavior Motifs --")
    if ir.get("behaviors"):
        for b in ir["behaviors"]:
            ev = "; ".join(b["evidence"][:3])
            lines.append(f"  [{b['score']:.2f}] {b['type']}  |  {ev}")
    else:
        lines.append("  (no significant behaviors detected)")

    iocs = ir.get("iocs", {})
    if any(iocs.values()):
        lines.append("\n-- IOCs --")
        for ioc_type, vals in iocs.items():
            if vals:
                lines.append(f"  {ioc_type}: {vals[:3]}")

    pg = ir.get("process_graph", {})
    if pg.get("nodes"):
        node_names = [n["name"] for n in pg["nodes"][:5]]
        lines.append(f"\n-- Process Graph --  nodes: {node_names}")
        if pg.get("edges"):
            for edge in pg["edges"][:3]:
                lines.append(f"  {edge['from_name']} -> {edge['to_name']}")

    lines.append("\n=== END IR ===")
    lines.append("\nTask: Generate a behavioral YARA rule that detects this malware based solely on the IR above.")
    return "\n".join(lines)


# ---- New IR-aware analysis function ----

IR_SYSTEM_PROMPT = """You are an expert malware analyst specializing in dynamic analysis and behavioral signatures.
You will receive a compact Behavior IR (Intermediate Representation) derived from a CAPEv2 sandbox report.
The IR contains scored behavior motifs, IOCs, and a process graph.
Your task is to produce a precise behavioral YARA detection rule.
Focus on the highest-scored motifs and concrete IOCs."""

IR_ANALYSIS_PROMPT_TEMPLATE = """
Analyze the following Behavior IR and provide:
1. **Behavior Summary**: What does this malware do? (2-3 sentences)
2. **Threat Classification**: e.g. trojan, ransomware, keylogger, banker, RAT
3. **Top Behavioral Indicators**: List the most important motifs and their significance
4. **MITRE ATT&CK Mapping**: Map the detected motifs to MITRE techniques
5. **YARA Detection Rule**: A behavioral rule using the IR's motifs and IOCs

{ir_prompt}

Provide your analysis in structured JSON format.
"""


def analyze_sample_ir(sample_hash: str, llm_client, ir_lookup: dict = None) -> dict:
    """
    LLM analysis using the Behavior IR instead of raw behavior text.
    Falls back to the legacy behavior_to_text approach if no IR is available.

    Args:
        sample_hash: SHA256 hash of the sample
        llm_client : LLM client instance
        ir_lookup  : dict mapping sha256 -> IR record (ir_by_hash)

    Returns:
        dict with analysis results, plus 'ir_used' flag
    """
    ir_lookup = ir_lookup or {}
    ir = ir_lookup.get(sample_hash)

    if ir:
        # Preferred path: use compact IR
        prompt = IR_ANALYSIS_PROMPT_TEMPLATE.format(ir_prompt=ir_to_prompt(ir))
        system = IR_SYSTEM_PROMPT
        analysis_source = "ir"
    else:
        # Fallback: legacy raw behavior text
        features_row = df_features[df_features['report_hash'] == sample_hash].iloc[0]
        details = next((d for d in behavior_details if d['report_hash'] == sample_hash), {})
        signature = next((s for s in signatures if s['report_hash'] == sample_hash), {})
        prompt = ANALYSIS_PROMPT_TEMPLATE.format(
            behavior_report=behavior_to_text(sample_hash, features_row, details, signature)
        )
        system = SYSTEM_PROMPT
        analysis_source = "raw_text"

    response = llm_client.generate(prompt, system_prompt=system, max_tokens=1500)

    try:
        import re as _re_parse
        json_match = _re_parse.search(r'\{[\s\S]*\}', response)
        analysis = json.loads(json_match.group()) if json_match else {'raw_response': response}
    except (json.JSONDecodeError, AttributeError):
        analysis = {'raw_response': response}

    analysis['sample_hash']    = sample_hash
    analysis['analysis_source'] = analysis_source
    if ir:
        analysis['top_motif']  = ir['behaviors'][0]['type'] if ir.get('behaviors') else 'none'
        analysis['family']     = ir.get('family', 'unknown')
    else:
        analysis['family'] = df_features[df_features['report_hash'] == sample_hash].iloc[0].get('family', 'unknown') if len(df_features[df_features['report_hash'] == sample_hash]) else 'unknown'

    return analysis


# Demo: show the IR prompt for one sample
if ir_by_hash:
    sample_hash_demo = next(iter(ir_by_hash))
    print("=" * 60)
    print("DEMO: IR-based prompt for one sample")
    print("=" * 60)
    print(ir_to_prompt(ir_by_hash[sample_hash_demo]))
else:
    print("No IR records loaded — run feature_extraction.ipynb or behavior_ir.ipynb first")

In [4]:
# Define prompt templates for different analysis tasks

SYSTEM_PROMPT = """You are an expert malware analyst specializing in dynamic analysis and behavioral signatures. 
Your task is to analyze malware behavior reports and generate actionable detection signatures.
Focus on identifying:
1. Malicious behavioral patterns
2. Indicators of Compromise (IOCs)
3. MITRE ATT&CK techniques
4. Detection rules and signatures
"""

ANALYSIS_PROMPT_TEMPLATE = """Analyze the following malware behavior report and provide:

1. **Behavior Summary**: A concise description of what this malware does
2. **Threat Classification**: Classify the threat type (ransomware, trojan, backdoor, etc.)
3. **Key Indicators**: List the most important behavioral indicators
4. **MITRE ATT&CK Mapping**: Map behaviors to MITRE techniques
5. **Detection Signature**: Generate a behavioral signature for detection

--- BEHAVIOR REPORT ---
{behavior_report}
--- END REPORT ---

Provide your analysis in structured JSON format.
"""

SIGNATURE_PROMPT_TEMPLATE = """Based on the following malware behavior data, generate a YARA-like behavioral detection rule.

The rule should:
1. Have a descriptive name based on the behavior
2. Include relevant API patterns
3. Include file/registry indicators if present
4. Include network indicators if present
5. Have appropriate severity rating

--- BEHAVIOR DATA ---
{behavior_report}
--- END DATA ---

Generate the detection rule in the following format:
```
rule <RULE_NAME> {{
    meta:
        description = "<description>"
        severity = "<low|medium|high|critical>"
        mitre = "<MITRE techniques>"
    
    behavior:
        <behavioral conditions>
    
    condition:
        <detection logic>
}}
```
"""

FAMILY_ANALYSIS_PROMPT = """Analyze the behavioral patterns across multiple samples from the {family_name} malware family.

Based on the following {num_samples} samples, identify:
1. **Common Behaviors**: Behaviors present in most/all samples
2. **Unique Family Indicators**: Behaviors specific to this family
3. **Variation Patterns**: How behaviors vary across samples
4. **Family Signature**: A signature that detects this family

--- SAMPLE BEHAVIORS ---
{sample_behaviors}
--- END SAMPLES ---

Provide a comprehensive family analysis.
"""

print("✓ Prompt templates defined")
print(f"\nSystem prompt ({len(SYSTEM_PROMPT)} chars)")
print(f"Analysis prompt template ({len(ANALYSIS_PROMPT_TEMPLATE)} chars)")
print(f"Signature prompt template ({len(SIGNATURE_PROMPT_TEMPLATE)} chars)")
print(f"Family analysis prompt ({len(FAMILY_ANALYSIS_PROMPT)} chars)")

✓ Prompt templates defined

System prompt (341 chars)
Analysis prompt template (543 chars)
Signature prompt template (691 chars)
Family analysis prompt (512 chars)


------------------------------------------------------------
## SECTION 5: LLM Integration
------------------------------------------------------------

Set up LLM client for behavior analysis. Supports multiple backends:

In [5]:
# LLM Configuration
# Available backends: 'ollama' (free local), 'huggingface' (free local), 'openai' (paid), 'mock' (testing)

# Auto-detect best available backend
def detect_llm_backend():
    """Auto-detect the best available LLM backend."""
    
    # 1. Try Ollama first (best free option)
    try:
        import requests
        response = requests.get('http://localhost:11434/api/tags', timeout=3)
        if response.status_code == 200:
            models = response.json().get('models', [])
            if models:
                print(f"✓ Ollama available with models: {[m['name'] for m in models]}")
                return 'ollama'
            else:
                print("⚠ Ollama running but no models. Run: ollama pull mistral")
    except:
        pass
    
    # 2. Check for OpenAI API key
    if os.environ.get('OPENAI_API_KEY'):
        print("✓ OpenAI API key found")
        return 'openai'
    
    # 3. Try HuggingFace (works but slower on CPU)
    try:
        import torch
        if torch.cuda.is_available():
            print("✓ GPU available - HuggingFace will work well")
            return 'huggingface'
        else:
            print("⚠ No GPU - HuggingFace will be slow, using mock instead")
    except:
        pass
    
    # 4. Fall back to mock
    print("ℹ Using mock backend for testing (fast, no external dependencies)")
    return 'mock'

# Auto-detect or manually set
LLM_BACKEND = detect_llm_backend()
# LLM_BACKEND = 'mock'  # Uncomment to force mock mode
# LLM_BACKEND = 'ollama'  # Uncomment after running: ollama pull mistral

# Ollama settings
OLLAMA_HOST = 'http://localhost:11434'
OLLAMA_MODEL = 'mistral'  # Good balance of quality and speed (7B params)

# HuggingFace settings
HF_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Small, fast, chat-optimized

print(f"\n✓ Selected LLM Backend: {LLM_BACKEND}")

# Instructions for setting up Ollama (after restart)
if LLM_BACKEND == 'mock':
    print("""
╔════════════════════════════════════════════════════════════════════╗
║  To use a real LLM (after restarting VS Code):                     ║
║                                                                    ║
║  1. Open a NEW terminal and run:                                   ║
║     ollama pull mistral                                            ║
║                                                                    ║
║  2. Then restart this notebook and it will auto-detect Ollama      ║
║                                                                    ║
║  For now, the mock backend will generate realistic test outputs.   ║
╚════════════════════════════════════════════════════════════════════╝
""")

✓ Ollama available with models: ['mistral:latest']

✓ Selected LLM Backend: ollama


In [9]:
import re
import time

class LLMClient:
    """Unified LLM client supporting multiple backends with retry logic."""
    
    def __init__(self, backend='mock'):
        self.backend = backend
        self.client = None
        self._setup_client()
    
    def _setup_client(self):
        """Initialize the appropriate LLM client."""
        if self.backend == 'openai':
            try:
                from openai import OpenAI
                self.client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
                print("✓ OpenAI client initialized")
            except ImportError:
                print("⚠ openai package not installed. Run: pip install openai")
                self.backend = 'mock'
        
        elif self.backend == 'ollama':
            try:
                import requests
                # Test connection and get available models
                response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
                if response.status_code == 200:
                    models = response.json().get('models', [])
                    model_names = [m['name'].split(':')[0] for m in models]
                    
                    # Select best available model
                    preferred = ['mistral', 'llama2', 'codellama', 'phi']
                    for pref in preferred:
                        if any(pref in m for m in model_names):
                            global OLLAMA_MODEL
                            OLLAMA_MODEL = next(m['name'] for m in models if pref in m['name'])
                            break
                    
                    print(f"✓ Ollama connected - using model: {OLLAMA_MODEL}")
                else:
                    print("⚠ Ollama not responding, falling back to mock")
                    self.backend = 'mock'
            except Exception as e:
                print(f"⚠ Ollama not available ({e}), falling back to mock")
                self.backend = 'mock'
        
        elif self.backend == 'huggingface':
            try:
                from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
                import torch
                
                print(f"Loading HuggingFace model: {HF_MODEL}...")
                print("(This may take a few minutes on first run)")
                
                device = 0 if torch.cuda.is_available() else -1
                
                self.client = pipeline(
                    'text-generation', 
                    model=HF_MODEL,
                    device=device,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
                )
                print(f"✓ HuggingFace pipeline initialized (device: {'GPU' if device == 0 else 'CPU'})")
            except Exception as e:
                print(f"⚠ HuggingFace failed: {e}")
                print("Falling back to mock backend")
                self.backend = 'mock'
        
        else:
            print("✓ Using mock LLM backend (generates realistic test outputs)")
    
    def generate(self, prompt: str, system_prompt: str = "", max_tokens: int = 1000) -> str:
        """Generate response from the LLM with retry logic."""
        
        if self.backend == 'openai':
            return self._generate_openai(prompt, system_prompt, max_tokens)
        elif self.backend == 'ollama':
            return self._generate_ollama_with_retry(prompt, system_prompt, max_tokens)
        elif self.backend == 'huggingface':
            return self._generate_hf(prompt, system_prompt, max_tokens)
        else:
            return self._generate_mock(prompt)
    
    def _generate_openai(self, prompt: str, system_prompt: str, max_tokens: int) -> str:
        """Generate using OpenAI API."""
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.7
        )
        return response.choices[0].message.content
    
    def _generate_ollama_with_retry(self, prompt: str, system_prompt: str, max_tokens: int, 
                                     max_retries: int = 3) -> str:
        """Generate using Ollama with retry logic and fallback to mock."""
        import requests
        
        # Reduce tokens for faster generation on CPU
        reduced_tokens = min(max_tokens, 500)
        
        for attempt in range(max_retries):
            try:
                print(f"  Ollama attempt {attempt + 1}/{max_retries}...", end=" ", flush=True)
                response = self._generate_ollama(prompt, system_prompt, reduced_tokens)
                print("✓")
                return response
            except requests.exceptions.Timeout:
                print(f"timeout", end="")
                if attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 30  # 30s, 60s, 90s
                    print(f", retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print("")
            except Exception as e:
                print(f"error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
        
        # Fallback to mock after all retries fail
        print("  ⚠ Ollama failed, using mock analysis")
        return self._generate_mock(prompt)
    
    def _generate_ollama(self, prompt: str, system_prompt: str, max_tokens: int) -> str:
        """Generate using local Ollama (single attempt)."""
        import requests
        
        # Truncate prompt if too long (keep under 2000 chars for faster processing)
        full_prompt = f"{system_prompt}\n\n{prompt}" if system_prompt else prompt
        if len(full_prompt) > 2500:
            full_prompt = full_prompt[:2500] + "\n...[truncated for speed]"
        
        response = requests.post(
            f"{OLLAMA_HOST}/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": full_prompt,
                "stream": False,
                "options": {
                    "num_predict": max_tokens,
                    "temperature": 0.7,
                    "num_ctx": 2048  # Smaller context for speed
                }
            },
            timeout=300  # 5 minutes per attempt
        )
        return response.json().get('response', '')
    
    def _generate_hf(self, prompt: str, system_prompt: str, max_tokens: int) -> str:
        """Generate using HuggingFace transformers."""
        if system_prompt:
            full_prompt = f"<|system|>\n{system_prompt}</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"
        else:
            full_prompt = f"<|user|>\n{prompt}</s>\n<|assistant|>\n"
        
        result = self.client(
            full_prompt, 
            max_new_tokens=min(max_tokens, 500),
            do_sample=True,
            temperature=0.7,
            pad_token_id=self.client.tokenizer.eos_token_id
        )
        generated = result[0]['generated_text']
        if '<|assistant|>' in generated:
            return generated.split('<|assistant|>')[-1].strip()
        return generated[len(full_prompt):]
    
    def _generate_mock(self, prompt: str) -> str:
        """Generate mock response for testing."""
        return self._create_mock_analysis(prompt)
    
    def _create_mock_analysis(self, prompt: str) -> str:
        """Create a realistic mock analysis based on prompt content."""
        
        has_network = 'network' in prompt.lower() and 'domains contacted: 0' not in prompt.lower()
        has_files = 'files written' in prompt.lower() and 'files written: 0' not in prompt.lower()
        has_registry = 'registry' in prompt.lower() and 'registry keys written: 0' not in prompt.lower()
        has_suspicious_apis = 'suspicious apis' in prompt.lower()
        has_injection = 'process injection' in prompt.lower() or 'WriteProcessMemory' in prompt
        
        family_match = re.search(r'Malware Family: (\w+)', prompt)
        family = family_match.group(1) if family_match else 'Unknown'
        
        threat_match = re.search(r'Threat Level: (\w+)', prompt)
        threat_level = threat_match.group(1) if threat_match else 'medium'
        
        analysis = {
            "behavior_summary": f"This sample belongs to the {family} malware family and exhibits ",
            "threat_classification": "trojan",
            "key_indicators": [],
            "mitre_techniques": [],
            "detection_signature": {}
        }
        
        behaviors = []
        
        if has_injection:
            behaviors.append("process injection capabilities")
            analysis['key_indicators'].append("Uses WriteProcessMemory/VirtualAllocEx for code injection")
            analysis['mitre_techniques'].append("T1055 - Process Injection")
            analysis['threat_classification'] = "trojan"
        
        if has_network:
            behaviors.append("command and control communication")
            analysis['key_indicators'].append("Establishes network connections to external servers")
            analysis['mitre_techniques'].append("T1071.001 - Application Layer Protocol: Web Protocols")
        
        if has_files:
            behaviors.append("file system manipulation")
            analysis['key_indicators'].append("Creates/modifies files on disk")
            analysis['mitre_techniques'].append("T1105 - Ingress Tool Transfer")
        
        if has_registry:
            behaviors.append("persistence via registry modification")
            analysis['key_indicators'].append("Modifies registry keys for persistence")
            analysis['mitre_techniques'].append("T1547.001 - Boot or Logon Autostart Execution: Registry Run Keys")
        
        if has_suspicious_apis:
            behaviors.append("suspicious API call patterns")
            analysis['key_indicators'].append("Calls known malicious Windows APIs")
            analysis['mitre_techniques'].append("T1106 - Native API")
        
        if not behaviors:
            behaviors.append("limited observable malicious activity")
            analysis['threat_classification'] = "potentially unwanted program"
        
        analysis['behavior_summary'] += ', '.join(behaviors) + '.'
        
        if has_injection and has_network:
            analysis['threat_classification'] = "backdoor/RAT"
        elif has_injection:
            analysis['threat_classification'] = "trojan"
        elif has_network and has_files:
            analysis['threat_classification'] = "downloader"
        
        analysis['detection_signature'] = {
            "name": f"{family}_Behavioral_Detection",
            "description": f"Detects {family} malware family based on behavioral patterns",
            "indicators": analysis['key_indicators'][:3],
            "mitre_mapping": analysis['mitre_techniques'][:3],
            "confidence": "high" if len(analysis['mitre_techniques']) >= 2 else "medium"
        }
        
        return json.dumps(analysis, indent=2)


# Initialize LLM client
llm = LLMClient(backend=LLM_BACKEND)
print(f"\n✓ LLM client ready (backend: {llm.backend})")

✓ Ollama connected - using model: mistral:latest

✓ LLM client ready (backend: ollama)


------------------------------------------------------------
## SECTION 6: Single Sample Analysis
------------------------------------------------------------

Analyze individual malware samples using the LLM:

In [10]:
def analyze_sample(sample_hash: str, llm_client: LLMClient) -> dict:
    """
    Perform LLM-based analysis on a single malware sample.
    
    Args:
        sample_hash: Hash of the sample to analyze
        llm_client: LLM client instance
    
    Returns:
        Dictionary containing LLM analysis results
    """
    
    # Get sample data
    features_row = df_features[df_features['report_hash'] == sample_hash].iloc[0]
    details = next((d for d in behavior_details if d['report_hash'] == sample_hash), {})
    signature = next((s for s in signatures if s['report_hash'] == sample_hash), {})
    
    # Convert to text
    behavior_text = behavior_to_text(sample_hash, features_row, details, signature)
    
    # Create analysis prompt
    prompt = ANALYSIS_PROMPT_TEMPLATE.format(behavior_report=behavior_text)
    
    # Get LLM analysis
    response = llm_client.generate(prompt, system_prompt=SYSTEM_PROMPT, max_tokens=1500)
    
    # Parse response
    try:
        # Try to extract JSON from response
        json_match = re.search(r'\{[\s\S]*\}', response)
        if json_match:
            analysis = json.loads(json_match.group())
        else:
            analysis = {'raw_response': response}
    except json.JSONDecodeError:
        analysis = {'raw_response': response}
    
    # Add metadata
    analysis['sample_hash'] = sample_hash
    analysis['family'] = features_row.get('family', 'unknown') if 'family' in features_row else 'unknown'
    analysis['original_threat_level'] = signature.get('threat_level', 'unknown')
    
    return analysis


# Test on a single sample
print("Analyzing a sample malware report...\n")

# Select a sample with activity
active_samples = df_features[df_features['num_api_calls'] > 10]
if len(active_samples) > 0:
    test_hash = active_samples.iloc[0]['report_hash']
else:
    test_hash = df_features.iloc[0]['report_hash']

analysis_result = analyze_sample_ir(test_hash, llm, ir_lookup=ir_by_hash)

print("=" * 60)
print("LLM ANALYSIS RESULT")
print("=" * 60)
print(json.dumps(analysis_result, indent=2))

Analyzing a sample malware report...

  Ollama attempt 1/3... 

timeout, retrying in 30s...
  Ollama attempt 2/3... ✓
LLM ANALYSIS RESULT
{
  "raw_response": " {\n  \"Behavior Summary\": \"The malware, identified as Qakbot, is a banker type malware that primarily focuses on API manipulation and process management for potential data theft, persistence, and command execution.\",\n\n  \"Threat Classification\": \"Banker\",\n\n  \"Key Indicators\": {\n    \"API Activity\": [\n      \"kernel32.dll.FlsAlloc\",\n      \"kernel32.dll.FlsGetValue\",\n      \"kernel32.dll.FlsSetValue\",\n      \"kernel32.dll.FlsFree\",\n      \"kernelbase.dll.InitializeCriticalSectionAndSpinCount\",\n      \"kernel32.dll.VirtualAlloc\",\n      \"kernel32.dll.LoadLibraryA\",\n      \"kernel32.dll.GetProcAddress\",\n      \"kernel32.dll.VirtualProtect\",\n      \"kernel32.dll.SetUnhandledExceptionFilter\"\n    ],\n    \"File System Activity\": {\n      \"Accessed Files\": 86,\n      \"Written Files\": 9,\n      \"Deleted Files\": 9\n    },\n    \"Registry Activity\": {\n      

------------------------------------------------------------
## SECTION 7: Batch Analysis
------------------------------------------------------------

Analyze multiple samples and aggregate results:

In [11]:
def batch_analyze(sample_hashes: list, llm_client: LLMClient, save_interval: int = 50) -> list:
    """
    Perform batch analysis on multiple samples.
    
    Args:
        sample_hashes: List of sample hashes to analyze
        llm_client: LLM client instance
        save_interval: Save intermediate results every N samples
    
    Returns:
        List of analysis results
    """
    
    all_results = []
    
    for idx, sample_hash in enumerate(tqdm(sample_hashes, desc="Analyzing samples")):
        try:
            result = analyze_sample_ir(sample_hash, llm_client, ir_lookup=ir_by_hash)
            all_results.append(result)
        except Exception as e:
            print(f"\nError analyzing {sample_hash[:16]}: {e}")
            all_results.append({
                'sample_hash': sample_hash,
                'error': str(e)
            })
        
        # Save intermediate results
        if (idx + 1) % save_interval == 0:
            intermediate_path = LLM_OUTPUT_DIR / f'analysis_checkpoint_{idx+1}.json'
            with open(intermediate_path, 'w') as f:
                json.dump(all_results, f, indent=2)
            print(f"\n✓ Saved checkpoint at {idx+1} samples")
    
    return all_results


# Analyze a subset of samples (adjust sample_count as needed)
SAMPLE_COUNT = 100  # Analyze first N samples (set to len(df_features) for all)

# Prioritize samples with more activity
df_sorted = df_features.sort_values('num_api_calls', ascending=False)
sample_hashes_to_analyze = df_sorted['report_hash'].head(SAMPLE_COUNT).tolist()

print(f"Starting batch analysis of {len(sample_hashes_to_analyze)} samples...\n")

# Run batch analysis
batch_results = batch_analyze(sample_hashes_to_analyze, llm)

print(f"\n✓ Completed analysis of {len(batch_results)} samples")

Starting batch analysis of 100 samples...



Analyzing samples:   0%|          | 0/100 [00:00<?, ?it/s]

  Ollama attempt 1/3... 

Analyzing samples:   1%|          | 1/100 [04:59<8:14:17, 299.57s/it]

✓
  Ollama attempt 1/3... timeout, retrying in 30s...
  Ollama attempt 2/3... 

...

✓
  Ollama attempt 1/3... 

Analyzing samples:  99%|█████████▉| 99/100 [5:06:38<02:52, 172.21s/it]

✓
  Ollama attempt 1/3... 

Analyzing samples: 100%|██████████| 100/100 [5:08:46<00:00, 185.26s/it]

✓

✓ Saved checkpoint at 100 samples

✓ Completed analysis of 100 samples


In [12]:
# Save batch results
batch_output_path = LLM_OUTPUT_DIR / 'llm_analysis_results.json'
with open(batch_output_path, 'w') as f:
    json.dump(batch_results, f, indent=2)
print(f"✓ Saved batch results to: {batch_output_path}")

# Create summary DataFrame
summary_rows = []
for result in batch_results:
    row = {
        'sample_hash': result.get('sample_hash', ''),
        'family': result.get('family', 'unknown'),
        'threat_classification': result.get('threat_classification', 'unknown'),
        'behavior_summary': result.get('behavior_summary', '')[:200] if isinstance(result.get('behavior_summary'), str) else '',
        'num_indicators': len(result.get('key_indicators', [])),
        'num_mitre': len(result.get('mitre_techniques', [])),
        'has_error': 'error' in result
    }
    summary_rows.append(row)

df_llm_summary = pd.DataFrame(summary_rows)
summary_csv_path = LLM_OUTPUT_DIR / 'llm_analysis_summary.csv'
df_llm_summary.to_csv(summary_csv_path, index=False)
print(f"✓ Saved summary to: {summary_csv_path}")

print(f"\nAnalysis Summary:")
print(f"  Total samples analyzed: {len(df_llm_summary)}")
print(f"  Successful analyses: {(~df_llm_summary['has_error']).sum()}")
print(f"  Errors: {df_llm_summary['has_error'].sum()}")

✓ Saved batch results to: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\llm_analysis\llm_analysis_results.json
✓ Saved summary to: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\llm_analysis\llm_analysis_summary.csv

Analysis Summary:
  Total samples analyzed: 100
  Successful analyses: 100
  Errors: 0


------------------------------------------------------------
## SECTION 8: Family-Level Analysis
------------------------------------------------------------

Analyze patterns across malware families:

In [ ]:
def analyze_family(family_name: str, llm_client: LLMClient, max_samples: int = 10) -> dict:
    """
    Analyze behavioral patterns for a malware family.
    
    Args:
        family_name: Name of the malware family
        llm_client: LLM client instance
        max_samples: Maximum samples to include in analysis
    
    Returns:
        Dictionary with family analysis results
    """
    
    # Get samples for this family
    if 'family' not in df_features.columns:
        return {'error': 'No family column in features'}
    
    family_samples = df_features[df_features['family'] == family_name]
    if len(family_samples) == 0:
        return {'error': f'No samples found for family {family_name}'}
    
    # Select representative samples (sorted by activity)
    selected = family_samples.sort_values('num_api_calls', ascending=False).head(max_samples)
    
    # Build behavior summaries for each sample
    sample_summaries = []
    for _, row in selected.iterrows():
        sample_hash = row['report_hash']
        details = next((d for d in behavior_details if d['report_hash'] == sample_hash), {})
        signature = next((s for s in signatures if s['report_hash'] == sample_hash), {})
        
        summary = f"Sample {sample_hash[:8]}:\n"
        summary += f"  - APIs: {row['num_api_calls']} ({row['unique_api_count']} unique)\n"
        summary += f"  - Files written: {row['num_write_files']}\n"
        summary += f"  - Registry writes: {row['num_write_registry']}\n"
        summary += f"  - Network: {row['num_domains']} domains, {row['num_http_requests']} HTTP\n"
        
        if details.get('suspicious_apis'):
            summary += f"  - Suspicious APIs: {', '.join(details['suspicious_apis'][:5])}\n"
        
        sample_summaries.append(summary)
    
    # Create family analysis prompt
    prompt = FAMILY_ANALYSIS_PROMPT.format(
        family_name=family_name,
        num_samples=len(selected),
        sample_behaviors='\n'.join(sample_summaries)
    )
    
    # Get LLM analysis
    response = llm_client.generate(prompt, system_prompt=SYSTEM_PROMPT, max_tokens=2000)
    
    return {
        'family': family_name,
        'total_samples': len(family_samples),
        'analyzed_samples': len(selected),
        'analysis': response
    }


# Analyze each family
family_analyses = {}

if 'family' in df_features.columns:
    families = df_features['family'].unique()
    print(f"Analyzing {len(families)} malware families...\n")
    
    for family in tqdm(families, desc="Analyzing families"):
        if pd.isna(family) or family == 'unknown':
            continue
        family_analyses[family] = analyze_family(family, llm)
    
    print(f"\n✓ Completed family analysis for {len(family_analyses)} families")
else:
    print("⚠ No family labels available in the dataset")

Analyzing 10 malware families...



Analyzing families:   0%|          | 0/10 [00:00<?, ?it/s]

  Ollama attempt 1/3... 

Analyzing families:  10%|█         | 1/10 [03:08<28:17, 188.64s/it]

✓
  Ollama attempt 1/3... 

...

✓
  Ollama attempt 1/3... 

Analyzing families:  40%|████      | 4/10 [13:01<19:36, 196.03s/it]

✓
  Ollama attempt 1/3... 

Analyzing families:  50%|█████     | 5/10 [15:49<15:30, 186.01s/it]

✓
  Ollama attempt 1/3... 

In [ ]:
# Save family analyses
if family_analyses:
    family_output_path = LLM_OUTPUT_DIR / 'family_analyses.json'
    with open(family_output_path, 'w') as f:
        json.dump(family_analyses, f, indent=2)
    print(f"✓ Saved family analyses to: {family_output_path}")
    
    # Display summary
    print(f"\n{'='*60}")
    print("FAMILY ANALYSIS SUMMARY")
    print(f"{'='*60}")
    
    for family, analysis in family_analyses.items():
        print(f"\n{family}:")
        print(f"  Total samples: {analysis.get('total_samples', 'N/A')}")
        print(f"  Analyzed: {analysis.get('analyzed_samples', 'N/A')}")
        # Print first 200 chars of analysis
        if 'analysis' in analysis:
            preview = analysis['analysis'][:200].replace('\n', ' ')
            print(f"  Analysis preview: {preview}...")

------------------------------------------------------------
## SECTION 9: Generate Detection Rules
------------------------------------------------------------

Generate YARA-like behavioral detection rules:

In [ ]:
def generate_detection_rule(sample_hash: str, llm_client: LLMClient) -> str:
    """
    Generate a behavioral detection rule for a sample.
    
    Args:
        sample_hash: Hash of the sample
        llm_client: LLM client instance
    
    Returns:
        Detection rule string
    """
    
    # Get sample data
    features_row = df_features[df_features['report_hash'] == sample_hash].iloc[0]
    details = next((d for d in behavior_details if d['report_hash'] == sample_hash), {})
    signature = next((s for s in signatures if s['report_hash'] == sample_hash), {})
    
    # Convert to text
    behavior_text = behavior_to_text(sample_hash, features_row, details, signature)
    
    # Create prompt
    prompt = SIGNATURE_PROMPT_TEMPLATE.format(behavior_report=behavior_text)
    
    # Generate rule
    response = llm_client.generate(prompt, system_prompt=SYSTEM_PROMPT, max_tokens=1000)
    
    return response


def generate_family_rule(family_name: str, llm_client: LLMClient) -> str:
    """
    Generate a family-wide detection rule.
    """
    
    if family_name not in family_analyses:
        return f"// No analysis available for {family_name}"
    
    analysis = family_analyses[family_name]
    
    prompt = f"""Based on the following family analysis, generate a YARA-like behavioral rule that detects the {family_name} malware family.

{analysis['analysis']}

Generate a comprehensive detection rule.
"""
    
    response = llm_client.generate(prompt, system_prompt=SYSTEM_PROMPT, max_tokens=1500)
    return response


# Generate rules for high-threat samples
print("Generating detection rules for high-threat samples...\n")

# Select high-threat samples
high_threat_results = [r for r in batch_results if r.get('original_threat_level') in ['high', 'medium']]
detection_rules = []

for result in tqdm(high_threat_results[:20], desc="Generating rules"):  # Limit to 20
    sample_hash = result.get('sample_hash')
    if sample_hash:
        try:
            rule = generate_detection_rule(sample_hash, llm)
            detection_rules.append({
                'sample_hash': sample_hash,
                'family': result.get('family', 'unknown'),
                'rule': rule
            })
        except Exception as e:
            print(f"Error generating rule for {sample_hash[:16]}: {e}")

print(f"\n✓ Generated {len(detection_rules)} detection rules")

In [ ]:
# Save detection rules
rules_output_path = LLM_OUTPUT_DIR / 'detection_rules.json'
with open(rules_output_path, 'w') as f:
    json.dump(detection_rules, f, indent=2)
print(f"✓ Saved detection rules to: {rules_output_path}")

# Also save as text file
rules_txt_path = LLM_OUTPUT_DIR / 'detection_rules.txt'
with open(rules_txt_path, 'w') as f:
    f.write("// Auto-generated Behavioral Detection Rules\n")
    f.write(f"// Generated: {pd.Timestamp.now()}\n")
    f.write(f"// Total rules: {len(detection_rules)}\n\n")
    
    for idx, rule_data in enumerate(detection_rules):
        f.write(f"// Rule {idx+1}: {rule_data['family']}\n")
        f.write(f"// Sample: {rule_data['sample_hash']}\n")
        f.write(rule_data['rule'])
        f.write("\n\n" + "="*60 + "\n\n")

print(f"✓ Saved rules as text to: {rules_txt_path}")

# Display first rule
if detection_rules:
    print(f"\n{'='*60}")
    print("SAMPLE DETECTION RULE")
    print(f"{'='*60}")
    print(f"Family: {detection_rules[0]['family']}")
    print(f"Sample: {detection_rules[0]['sample_hash'][:32]}...")
    print(f"\n{detection_rules[0]['rule'][:500]}...")

------------------------------------------------------------
## SECTION 10: Summary and Visualization
------------------------------------------------------------

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Create summary visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LLM Behavior Analysis Summary', fontsize=16, fontweight='bold')

# 1. Samples analyzed per family
if 'family' in df_llm_summary.columns:
    family_counts = df_llm_summary['family'].value_counts().head(10)
    axes[0, 0].barh(family_counts.index, family_counts.values, color='steelblue', edgecolor='black')
    axes[0, 0].set_xlabel('Sample Count')
    axes[0, 0].set_title('Samples Analyzed per Family (Top 10)')
    axes[0, 0].invert_yaxis()

# 2. Threat classification distribution
if 'threat_classification' in df_llm_summary.columns:
    threat_counts = df_llm_summary['threat_classification'].value_counts()
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(threat_counts)))
    axes[0, 1].pie(threat_counts, labels=threat_counts.index, autopct='%1.1f%%', colors=colors)
    axes[0, 1].set_title('Threat Classification Distribution')

# 3. MITRE techniques coverage
if 'num_mitre' in df_llm_summary.columns:
    axes[1, 0].hist(df_llm_summary['num_mitre'], bins=10, color='coral', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Number of MITRE Techniques')
    axes[1, 0].set_ylabel('Sample Count')
    axes[1, 0].set_title('MITRE Technique Coverage per Sample')
    axes[1, 0].axvline(df_llm_summary['num_mitre'].mean(), color='red', linestyle='--', label=f'Mean: {df_llm_summary["num_mitre"].mean():.1f}')
    axes[1, 0].legend()

# 4. Indicators per sample
if 'num_indicators' in df_llm_summary.columns:
    axes[1, 1].hist(df_llm_summary['num_indicators'], bins=10, color='teal', edgecolor='black', alpha=0.7)
    axes[1, 1].set_xlabel('Number of Key Indicators')
    axes[1, 1].set_ylabel('Sample Count')
    axes[1, 1].set_title('Key Indicators Identified per Sample')
    axes[1, 1].axvline(df_llm_summary['num_indicators'].mean(), color='red', linestyle='--', label=f'Mean: {df_llm_summary["num_indicators"].mean():.1f}')
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig(LLM_OUTPUT_DIR / 'llm_analysis_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization to: {LLM_OUTPUT_DIR / 'llm_analysis_summary.png'}")

In [ ]:
# Final summary
print("\n" + "="*70)
print("LLM BEHAVIOR ANALYSIS - FINAL SUMMARY")
print("="*70)

print(f"\n1. ANALYSIS OVERVIEW")
print(f"   Total samples analyzed: {len(batch_results)}")
print(f"   Successful analyses: {len([r for r in batch_results if 'error' not in r])}")
print(f"   Families analyzed: {len(family_analyses)}")
print(f"   Detection rules generated: {len(detection_rules)}")

print(f"\n2. LLM CONFIGURATION")
print(f"   Backend: {LLM_BACKEND}")
print(f"   Model: {'gpt-4o-mini' if LLM_BACKEND == 'openai' else OLLAMA_MODEL if LLM_BACKEND == 'ollama' else 'mock'}")

print(f"\n3. OUTPUT FILES")
print(f"   • {LLM_OUTPUT_DIR / 'llm_analysis_results.json'}")
print(f"   • {LLM_OUTPUT_DIR / 'llm_analysis_summary.csv'}")
print(f"   • {LLM_OUTPUT_DIR / 'family_analyses.json'}")
print(f"   • {LLM_OUTPUT_DIR / 'detection_rules.json'}")
print(f"   • {LLM_OUTPUT_DIR / 'detection_rules.txt'}")
print(f"   • {LLM_OUTPUT_DIR / 'llm_analysis_summary.png'}")

print(f"\n4. NEXT STEPS")
print(f"   • Review generated detection rules")
print(f"   • Fine-tune prompts for better signatures")
print(f"   • Validate rules against known samples")
print(f"   • Integrate with YARA/CAPE signature format")

print("\n" + "="*70)